In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier
)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [ ]:
df_clf = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df_clf.head()

# Pre- Processing

In [ ]:
df_clf.describe()

In [ ]:
df_clf.isnull().sum()

In [ ]:
df_clf.info()

In [ ]:
# cleaning dataset
# irrelevant column: PassengerId, Name, Cabin (too much null value), ticket (unique irrelavent values)
df_clf = df_clf[['Survived','Pclass','Sex','Age','SibSp','Parch','Fare']]
df_clf['Age'] = df_clf['Age'].fillna(df_clf['Age'].mean())
df_clf['Sex'] = df_clf['Sex'].map({'male':0, 'female':1})

In [ ]:
df_clf.info()

In [ ]:
df_clf.head()

In [ ]:
df_clf['Pclass'].unique()

In [ ]:
df_clf['SibSp'].unique()

In [ ]:
df_clf['Parch'].unique()

In [ ]:
df_clf['Sex'].unique()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# train - test split
X_clf = df_clf.drop('Survived', axis=1)
y_clf = df_clf['Survived']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

num_cols = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']

In [ ]:

# 1. Scale the numerical columns directly in the dataframe
scaler = StandardScaler()

# Fit and transform the training set
X_train_c[num_cols] = scaler.fit_transform(X_train_c[num_cols])

# Only transform the test set (to avoid data leakage)
X_test_c[num_cols] = scaler.transform(X_test_c[num_cols])

# 2. Sex is already 0/1, so we do nothing to it!
# Your data is now ready.

In [ ]:
# Classification Evaluation Function
def evaluate_classification(model):
    model.fit(X_train_c, y_train_c)
    y_pred = model.predict(X_test_c)
    
    print("Accuracy:", accuracy_score(y_test_c, y_pred))
    print("Precision:", precision_score(y_test_c, y_pred))
    print("Recall:", recall_score(y_test_c, y_pred))
    print("F1 Score:", f1_score(y_test_c, y_pred))
    
    print("\nConfusion Matrix:\n", confusion_matrix(y_test_c, y_pred))
    print("\nClassification Report:\n", classification_report(y_test_c, y_pred))
    print("-"*40)

# Logistic Regression (L2)

In [ ]:
# Logistic Regression (L2)
log_reg_l2 = LogisticRegression(penalty='l2', solver='lbfgs')
evaluate_classification(log_reg_l2)

# Logistic Regression (L1)

In [ ]:
# Logistic Regression (L1)
log_reg_l1 = LogisticRegression(penalty='l1', solver='liblinear')
evaluate_classification(log_reg_l1)

# ElasticNet Logistic

In [ ]:
# ElasticNet Logistic
log_reg_en = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio=0.5
)
evaluate_classification(log_reg_en)

In [ ]:
# ROC Curve
log_reg_l2.fit(X_train_c, y_train_c)
y_prob = log_reg_l2.predict_proba(X_test_c)[:,1]

fpr, tpr, _ = roc_curve(y_test_c, y_prob)
auc = roc_auc_score(y_test_c, y_prob)

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

# KNN

In [ ]:
# KNN – Hyperparameter Tuning (k from 1 to 15)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

k_values = range(1, 16)
accuracy_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_c, y_train_c)
    y_pred = knn.predict(X_test_c)
    
    acc = accuracy_score(y_test_c, y_pred)
    accuracy_scores.append(acc)

# Plot graph
plt.figure(figsize=(8,5))
plt.plot(k_values, accuracy_scores, marker='o')
plt.xlabel("Value of K")
plt.ylabel("Accuracy")
plt.title("K vs Accuracy")
plt.xticks(k_values)
plt.grid(True)
plt.show()

In [ ]:
# Select Best K
best_k = k_values[accuracy_scores.index(max(accuracy_scores))]
print("Best K:", best_k)
print("Best Accuracy:", max(accuracy_scores))

In [ ]:
# Train Final Model with Best K
final_knn = KNeighborsClassifier(n_neighbors=best_k)
evaluate_classification(final_knn)

# Gaussian Naive Bayes

In [ ]:
# Gaussian Naive Bayes
gnb = GaussianNB()
evaluate_classification(gnb)

# SGD Classifier

In [ ]:
# SGD Classifier
sgd_clf = SGDClassifier(loss='log_loss', max_iter=1000)
evaluate_classification(sgd_clf)

# SVM

In [ ]:
from sklearn.svm import SVC

# Train SVM on full features
svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
evaluate_classification(svm_clf)

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize Decision Tree
# max_depth=5 helps prevent the tree from memorizing the training data (overfitting)
dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)

# Evaluate using your function
evaluate_classification(dt_clf)

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest
# n_estimators=100 is the number of trees in the forest
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=7, random_state=42)

# Evaluate using your function
evaluate_classification(rf_clf)

# Random Forest without scaling

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest
# n_estimators=100 is the number of trees in the forest
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=7, random_state=42)

# Evaluate using your function
evaluate_classification(rf_clf)